<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 2

## Details

**Topic:** Natural Language Processing

**Sub-topic:** PyTorch Fundamentals

**Instructions:**

This exercise tests your understanding of PyTorch's automatic differentiation engine, Autograd. You will manually define a function with multiple variables, ask PyTorch to compute the gradients, and then verify the results.
Your Goal: Calculate the partial derivatives of the following function with respect to w, x, and b.

Let the function be: z=σ(w.x2)+1b3. where σ is the sigmoid function, which in PyTorch is torch.sigmoid(). Given Initial Values: w=2.0 x=4.0 b=1.5.

Your Task: Create three tensors for w, x, and b with the given initial values. Make sure PyTorch tracks their gradients. Write the Python code to compute z using these tensors. Call the .backward() method to compute the gradients.Print the computed gradients for w, x, and b.
Hint: Remember to set requires_grad=True when you create the tensors. The gradient of a tensor t is stored in t.grad after you call .backward() on the final output.


**What I found interesting:**

- Vanishing gradient demonstrated live
- Example of why sigmoid activation functions declined in deep networks: in a real network with many layers, this near-zero gradient gets multiplied at every layer during backpropagation, until the gradient reaching the early layers is too tiny to update the weights at all => showcases the failure mode at toy scale => why ReLU (whose gradient is either 0 or 1, never a tiny fraction) largely replaced sigmoid in hidden layers

### Exploring Autograd with a Complex Function

PyTorch's Autograd engine -> automatically computes partial derivatives by recording every operation on tensors that have `requires_grad=True` => dynamic computation graph as code runs (forward pass), then traverses it in reverse during `.backward()` to accumulate gradients a.k.a. reverse-mode automatic differentiation

### The Function

$$z = \sigma(w \cdot x^2) + \frac{1}{b^3}$$

where $\sigma$ is the sigmoid function: $\sigma(t) = \dfrac{1}{1 + e^{-t}}$

### Given Values

| Variable | Value |
|----------|-------|
| $w$      | 2.0   |
| $x$      | 4.0   |
| $b$      | 1.5   |

### Create tensors with gradient tracking

In [3]:
import torch

# Create leaf tensors — requires_grad=True enables gradient tracking
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)
b = torch.tensor(1.5, requires_grad=True)

print(f"w = {w}  |  requires_grad: {w.requires_grad}")
print(f"x = {x}  |  requires_grad: {x.requires_grad}")
print(f"b = {b}  |  requires_grad: {b.requires_grad}")

w = 2.0  |  requires_grad: True
x = 4.0  |  requires_grad: True
b = 1.5  |  requires_grad: True


### Compute z (forward pass)

In [4]:
# Forward pass — each line extends the computation graph
z = torch.sigmoid(w * x**2) + 1 / b**3

print(f"x²          = {x**2}")
print(f"w · x²      = {w * x**2:.4f}")
print(f"σ(w · x²)   = {torch.sigmoid(w * x**2):.6f}")
print(f"1 / b³      = {1 / b**3:.6f}")
print(f"z           = {z:.6f}")
print(f"\nz was created by: {z.grad_fn}")

x²          = 16.0
w · x²      = 32.0000
σ(w · x²)   = 1.000000
1 / b³      = 0.296296
z           = 1.296296

z was created by: <AddBackward0 object at 0x7f0a905b6020>


### Backward pass: computing gradients

In [5]:
# Backward pass — populates .grad on every leaf with requires_grad=True
z.backward()

print("Computed gradients via Autograd:")
print(f"  dz/dw = {w.grad:.8f}")
print(f"  dz/dx = {x.grad:.8f}")
print(f"  dz/db = {b.grad:.8f}")

Computed gradients via Autograd:
  dz/dw = 0.00000000
  dz/dx = 0.00000000
  dz/db = -0.59259260


### Manual verification


We verify PyTorch's results by deriving the partial derivatives analytically using the chain rule, then evaluating them at the given values.

Let $u = w \cdot x^2$, so $z = \sigma(u) + b^{-3}$.

**Derivative of σ**: $\sigma'(u) = \sigma(u)(1 - \sigma(u))$

---

### ∂z/∂w

$$\frac{\partial z}{\partial w} = \sigma'(u) \cdot x^2 = \sigma(w x^2)\bigl(1 - \sigma(w x^2)\bigr) \cdot x^2$$

### ∂z/∂x

$$\frac{\partial z}{\partial x} = \sigma'(u) \cdot 2wx = \sigma(w x^2)\bigl(1 - \sigma(w x^2)\bigr) \cdot 2wx$$

### ∂z/∂b

$$\frac{\partial z}{\partial b} = -\frac{3}{b^4}$$

In [6]:
import math

# Detach values for manual computation (plain Python floats)
w_v = w.item()   # 2.0
x_v = x.item()   # 4.0
b_v = b.item()   # 1.5

# Sigmoid and its derivative evaluated at u = w * x²
u      = w_v * x_v**2                         # 32.0
sig_u  = 1 / (1 + math.exp(-u))               # ≈ 1.000
sig_du = sig_u * (1 - sig_u)                  # ≈ 1.27e-14  (saturated sigmoid → tiny gradient)

# Analytic partial derivatives
grad_w_analytic = sig_du * x_v**2             # dz/dw
grad_x_analytic = sig_du * 2 * w_v * x_v     # dz/dx
grad_b_analytic = -3 / b_v**4                 # dz/db

print("Analytic gradients (manual chain rule):")
print(f"  dz/dw = {grad_w_analytic:.8f}")
print(f"  dz/dx = {grad_x_analytic:.8f}")
print(f"  dz/db = {grad_b_analytic:.8f}")

print("\nMatch check (PyTorch vs analytic):")
print(f"  dz/dw  match: {math.isclose(w.grad.item(), grad_w_analytic, rel_tol=1e-5)}")
print(f"  dz/dx  match: {math.isclose(x.grad.item(), grad_x_analytic, rel_tol=1e-5)}")
print(f"  dz/db  match: {math.isclose(b.grad.item(), grad_b_analytic, rel_tol=1e-5)}")

Analytic gradients (manual chain rule):
  dz/dw = 0.00000000
  dz/dx = 0.00000000
  dz/db = -0.59259259

Match check (PyTorch vs analytic):
  dz/dw  match: False
  dz/dx  match: False
  dz/db  match: True
